# SHIPP — Unit test: delete-aware current-state logic

Tests `reconstruct_current_state` and `independent_current_keys` from `common/shipp_silver_lib`
on a **synthetic** CDC history. No Lakebase, no real tables, nothing is written.

Run it any time, as often as you like: **Clear state and outputs → Run all**.
Run it again after *any* change to `shipp_silver_lib`.

| Key | History | Expected |
|---|---|---|
| L1 | insert → delete → insert → delete (the real `delete-test-listing-001` history) | absent |
| L2 | insert → update (pre + post image) | present, `update_postimage` |
| L3 | insert → delete → insert | present, `insert` |
| L4 | insert → update, both in one transaction (same `_pg_lsn`) | present, `update_postimage` (tie broken by `_sort_by`) |

In [0]:
%run ../common/shipp_silver_lib

In [0]:
rows = [
    # listing_id, _pg_change_type,   _pg_lsn, _sort_by, status
    ("L1", "insert",            100,  8, "AVAILABLE"),
    ("L1", "delete",            101,  9, "AVAILABLE"),
    ("L1", "insert",            102, 10, "AVAILABLE"),
    ("L1", "delete",            103, 11, "AVAILABLE"),

    ("L2", "insert",            100,  0, "DRAFT"),
    ("L2", "update_preimage",   104,  4, "DRAFT"),
    ("L2", "update_postimage",  104,  5, "AVAILABLE"),

    ("L3", "insert",            100,  1, "AVAILABLE"),
    ("L3", "delete",            105, 12, "AVAILABLE"),
    ("L3", "insert",            106, 13, "AVAILABLE"),

    ("L4", "insert",            200, 20, "DRAFT"),
    ("L4", "update_preimage",   200, 21, "DRAFT"),
    ("L4", "update_postimage",  200, 22, "AVAILABLE"),
]
fixture_df = spark.createDataFrame(
    rows, "listing_id string, _pg_change_type string, _pg_lsn long, _sort_by long, status string"
)

latest_events_df, current_rows_df, deleted_keys_df = reconstruct_current_state(fixture_df, "listing_id")

current = {r["listing_id"]: (r["_pg_change_type"], r["status"]) for r in current_rows_df.collect()}
deleted = {r["listing_id"] for r in deleted_keys_df.collect()}
independent = independent_current_keys(fixture_df, "listing_id")

print("current_rows:", current)
print("deleted:     ", deleted)
print("independent: ", sorted(independent))

assert current == {
    "L2": ("update_postimage", "AVAILABLE"),
    "L3": ("insert", "AVAILABLE"),
    "L4": ("update_postimage", "AVAILABLE"),
}, f"FAILED: unexpected current state {current}"
assert deleted == {"L1"}, f"FAILED: unexpected deleted keys {deleted}"
assert independent == set(current), "FAILED: window logic and max_by logic disagree"
print("PASS — delete-aware reconstruction is correct on the synthetic history.")

### Regression proof

The old logic (filter out deletes **before** ranking) must fail this fixture by resurrecting `L1`.
If this cell ever fails, the fixture no longer detects the original bug.

In [0]:
from pyspark.sql.window import Window

old_window = Window.partitionBy("listing_id").orderBy(*cdc_newest_first())
old_current = {
    r["listing_id"]
    for r in (
        fixture_df
        .filter(F.col("_pg_change_type").isin("insert", "update_postimage"))
        .withColumn("_rn", F.row_number().over(old_window))
        .filter("_rn = 1")
        .collect()
    )
}
print("old logic current keys:", sorted(old_current))
assert "L1" in old_current, "Fixture no longer detects the resurrection bug."
print("PASS — fixture detects the old resurrection bug (old logic keeps L1).")